# Text Preprocessing and Tokenization

Every NLP pipeline starts with turning raw text into structured features. This notebook covers:
1. **Basic preprocessing** -- lowering, punctuation removal, stopwords
2. **Tokenization** with spaCy
3. **Lemmatization** vs stemming
4. **Named Entity Recognition** (NER)

In [ ]:
import re
import numpy as np
from collections import Counter

try:
    import spacy
    nlp = spacy.load('en_core_web_sm')
    HAS_SPACY = True
except (ImportError, OSError):
    HAS_SPACY = False
    print('spaCy not available. Install: pip install spacy && python -m spacy download en_core_web_sm')

try:
    import nltk
    from nltk.tokenize import word_tokenize
    from nltk.stem import PorterStemmer, WordNetLemmatizer
    HAS_NLTK = True
except ImportError:
    HAS_NLTK = False
    print('nltk not installed -- pip install nltk')

In [ ]:
# Sample corpus
texts = [
    "The Topological Data Analysis (TDA) framework provides robust tools for shape analysis.",
    "Dr. Smith published 3 papers on persistent homology in 2023.",
    "Machine learning models can't handle high-dimensional data without preprocessing.",
    "Natural Language Processing has been revolutionised by transformer architectures."
]
print(f"{len(texts)} sample texts loaded.")

## 1. Basic Preprocessing

Common steps: lowercasing, removing punctuation, removing stopwords, removing numbers.

In [ ]:
def basic_preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)       # remove non-alpha
    tokens = text.split()
    # Simple stopword list
    stopwords = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'in', 'on', 'for',
                 'by', 'has', 'have', 'been', 'can', 'of', 'to', 'and', 'with', 'without'}
    tokens = [t for t in tokens if t not in stopwords and len(t) > 1]
    return tokens

for text in texts:
    tokens = basic_preprocess(text)
    print(f"Original: {text[:60]}...")
    print(f"Tokens:   {tokens}\n")

## 2. Tokenization with spaCy

spaCy provides **linguistic tokenization** that handles contractions, abbreviations, and special cases correctly.

In [ ]:
if HAS_SPACY:
    doc = nlp(texts[2])  # "Machine learning models can't handle..."
    print(f"{'Token':<15} {'Lemma':<15} {'POS':<8} {'Is stop?':<8}")
    print('-' * 50)
    for token in doc:
        print(f"{token.text:<15} {token.lemma_:<15} {token.pos_:<8} {token.is_stop}")

## 3. Stemming vs Lemmatization

- **Stemming** (Porter): crude suffix stripping -- fast but can produce non-words.
- **Lemmatization** (spaCy/WordNet): maps to dictionary form -- slower but linguistically correct.

In [ ]:
words = ['running', 'ran', 'better', 'studies', 'computing', 'revolutionised']

if HAS_NLTK:
    stemmer = PorterStemmer()
    print(f"{'Word':<18} {'Stem':<15}")
    for w in words:
        print(f"{w:<18} {stemmer.stem(w):<15}")

if HAS_SPACY:
    print(f"\n{'Word':<18} {'Lemma':<15}")
    for w in words:
        doc = nlp(w)
        print(f"{w:<18} {doc[0].lemma_:<15}")

## 4. Named Entity Recognition (NER)

NER identifies proper nouns, dates, organisations, etc.

In [ ]:
if HAS_SPACY:
    text = "Dr. Smith published 3 papers on persistent homology at MIT in 2023."
    doc = nlp(text)
    print(f"Text: {text}\n")
    print(f"{'Entity':<25} {'Label':<12} {'Description'}")
    print('-' * 60)
    for ent in doc.ents:
        print(f"{ent.text:<25} {ent.label_:<12} {spacy.explain(ent.label_)}")

In [ ]:
# Build a simple vocabulary from corpus
if HAS_SPACY:
    all_tokens = []
    for text in texts:
        doc = nlp(text)
        tokens = [t.lemma_.lower() for t in doc if not t.is_stop and not t.is_punct and len(t) > 1]
        all_tokens.extend(tokens)
    
    vocab = Counter(all_tokens)
    print(f"Vocabulary size: {len(vocab)}")
    print(f"\nTop 10 tokens: {vocab.most_common(10)}")

## Key Takeaways

- **Preprocessing** choices (lowering, stopwords, lemmatization) directly impact downstream models.
- **spaCy** provides an efficient, industrial-strength NLP pipeline.
- **Lemmatization** is generally preferred over stemming for modern NLP.
- Modern transformers (BERT, GPT) use **subword tokenization** (BPE, WordPiece) and often skip traditional preprocessing.

**Next:** Word embeddings (Word2Vec, TF-IDF).